# 使用 Primus 训练模型
**作者**：Wei Cai
**知识水平**：中级

本教程演示如何利用 Primus 在 AMD GPU 上使用 ROCm 预训练 [Qwen2.5-7B](https://huggingface.co/Qwen/Qwen2.5-7B) 大语言模型 (LLM)。 Primus 是适用于 AMD Instinct™ GPU 的统一且灵活的训练框架，旨在支持包括 Megatron 在内的多个训练引擎后端，以提供可扩展的高性能模型训练。性能加速由 [Primus-Turbo](https://github.com/AMD-AGI/Primus-Turbo) 和 ROCm 库提供支持。

Primus with Megatron 旨在取代 [ROCm Megatron-LM 训练](https://rocm.docs.amd.com/en/docs-7.2.4/how-to/rocm-for-ai/training/benchmark-docker/megatron-lm.html) 工作流程。要了解如何使用 Megatron 将工作负载从 Megatron-LM 迁移到 Primus，请参阅[将工作负载从 Megatron-LM 迁移到 Primus（Megatron 后端）](https://rocm.docs.amd.com/en/docs-7.2.4/how-to/rocm-for-ai/training/benchmark-docker/previous-versions/megatron-lm-primus-migration-guide.html)。

通过学习本教程，您将了解如何使用 Primus 设置和运行训练、利用其性能优化，并获得使用 Primus 简化模型训练工作流程的实践经验。

## 前置条件

本教程是使用以下设置进行开发和测试的。 

### 操作系统

* **Ubuntu 22.04**：确保您的系统运行的是 Ubuntu 22.04。

### 硬件

* **AMD Instinct™ GPU**：本教程在 AMD Instinct MI300X GPU 的完整节点（八个 MI300X GPU）上进行了测试。确保您使用的是 AMD Instinct GPU 或支持 ROCm 的兼容硬件，并且您的系统满足[官方要求](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/reference/system-requirements.html)。

### 软件

* **ROCm 7.0.0**：按照 [ROCm 安装指南](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) 安装并验证 ROCm。安装后，使用以下命令确认您的设置：

    ``` bash
    amd-smi
    ```

    此命令列出了您的 AMD GPU 以及相关详细信息。
    
    **注意**：对于 ROCm 6.4 及更早版本，请改用 `rocm-smi` 命令。

* **Docker**：确保 Docker 已正确安装和配置。请遵循适用于您的操作系统的 Docker 安装指南。

   **注意**：确保正确配置 Docker 权限。要配置权限以允许非 root 访问，请运行以下命令：

   ``` bash
   sudo usermod -aG docker $USER
   newgrp docker
   ```

   验证 Docker 是否正常工作：

   ``` bash
   docker run hello-world
   ```


## 系统验证

在运行 AI 工作负载之前，务必确保您的 AMD 硬件配置正确且性能最佳。

一般来说，禁用 NUMA（非统一内存访问）自动平衡可以提高应用程序性能。但是，此设置可能会损害某些类型的工作负载的性能。

运行以下命令来验证当前的 NUMA 设置：

``` bash
cat /proc/sys/kernel/numa_balancing
``` 

输出 `0` 表示 NUMA 自动平衡已禁用。如果没有任何输出或者输出为`1`，请运行以下命令禁用NUMA自动平衡。

``` bash
sudo sh -c 'echo 0 > /proc/sys/kernel/numa_balancing'
``` 

有关详细信息，请参阅[禁用 NUMA 自动平衡](https://instinct.docs.amd.com/projects/amdgpu-docs/en/latest/system-optimization/mi300x.html#disable-numa-auto-balancing)。

## 设置环境

请按照以下步骤准备培训环境。

### 1. 拉取 Docker 镜像

确保您的系统满足[系统要求](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/reference/system-requirements.html)。

拉取本教程所需的 Docker 映像：

``` bash
docker pull rocm/primus:v25.9_gfx942
```

### 2.启动Docker容器

启动 Docker 容器并映射必要的目录。 

``` bash
docker run -it \
    --device /dev/dri \
    --device /dev/kfd \
    --device /dev/infiniband \
    --network host --ipc host \
    --group-add video \
    --cap-add SYS_PTRACE \
    --security-opt seccomp=unconfined \
    --privileged \
    --shm-size 128G \
    --name primus_training_env \
    rocm/primus:v25.9_gfx942
```

**注意**：如果退出后需要返回到 `primus_training_env` 容器，请使用以下命令：

``` bash
docker start primus_training_env
docker exec -it primus_training_env bash
```

**注意**：确保Notebook文件在启动后复制到 `/workspace` 目录或上传到 Jupyter Notebook 环境中。保存终端输出中提供的令牌或 URL，以便从 Web 浏览器访问Notebook。您可以从 [AI Developer Hub GitHub 存储库](https://github.com/ROCm/gpuaidev) 下载此Notebook。


### 3.安装并启动 Jupyter

在 Docker 容器内，使用以下命令安装 Jupyter：

``` bash
pip install jupyter
```

启动 Jupyter 服务器：

``` bash
jupyter-lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root
```

**注意**：在运行上述命令之前，请确保您的系统上尚未使用端口 `8888`。如果是，您可以通过将 `--port=8888` 替换为其他端口号来指定其他端口，例如 `--port=8890`。

## 编辑训练配置

Primus 在 YAML 中为 `examples/megatron/configs` 中的每个模型定义了训练配置。

例如，要更新 Qwen2.5-7B 的训练参数，请更新 `examples/megatron/configs/primus_qwen2.5_7B-pretrain.yaml`。其他模型的 YAML 训练配置文件遵循此命名约定。

### 数据集选项

您可以使用模拟数据或真实数据进行训练。

- 模拟数据可用于测试和验证。使用 `mock_data` 字段在模拟数据和真实数据之间切换。默认值为 `true`（启用）。

  ``` yaml
  mock_data: true
  ```

- 如果您使用的是真实数据集，请更新 `train_data_path` 字段以指向您的数据集位置。

  ``` yaml
  mock_data: false
  train_data_path: /path/to/your/dataset
  ```

确保可在 Docker 容器内访问这些文件。

In [ ]:
mock_data = True   # Set to False if you want to use a real dataset
train_data_path = "/path/to/your/dataset"  # Only used if mock_data=False

# Conditional logic
if mock_data:
    print("✅ Using mock dataset for testing and validation.")
    # Insert mock dataset loading logic here
else:
    print(f"✅ Using real dataset from: {train_data_path}")
    # Example: verify the dataset path exists
    import os
    if os.path.exists(train_data_path):
        print("📂 Dataset found and accessible.")
        # Insert real dataset loading logic here
    else:
        print("❌ Dataset path not found! Please check your configuration.")

## 运行训练
使用以下示例命令设置环境、配置[关键选项](https://rocm.docs.amd.com/en/docs-7.2.4/how-to/rocm-for-ai/training/benchmark-docker/previous-versions/primus-megatron-v25.8.html?model=primus_pyt_megatron_lm_train_qwen2.5-7b)，并使用 Primus 和 Megatron 后端在 AMD Instinct GPU 上运行训练。

### 单节点训练
要在单个节点上运行训练，请导航到 `/workspace/Primus` 并使用以下部分中的设置命令：

In [ ]:
%cd /workspace/Primus

#### 1.设置环境变量

运行以下命令设置并确认环境变量。

In [ ]:
# Install dependencies
!pip install -r requirements.txt

# Set environment variables for current process
import os
os.environ["HSA_NO_SCRATCH_RECLAIM"] = "1"
os.environ["NVTE_CK_USES_BWD_V3"] = "1"

# (Optional) Print to confirm they are set
print("HSA_NO_SCRATCH_RECLAIM =", os.environ.get("HSA_NO_SCRATCH_RECLAIM"))
print("NVTE_CK_USES_BWD_V3 =", os.environ.get("NVTE_CK_USES_BWD_V3"))

设置完成后，运行适当的训练命令。以下运行命令是针对Qwen2.5-7B定制的。请参阅[支持的型号](https://rocm.docs.amd.com/projects/primus/en/latest/06-developer-guide/model-support-matrix.html) 以切换到其他可用型号。

#### 2.显示训练配置

In [ ]:
# Display the training config YAML file
import yaml
from IPython.display import Markdown, display

with open("examples/megatron/configs/qwen2.5_7B-pretrain.yaml", "r") as f:
    train_config = yaml.safe_load(f)

yaml_text = yaml.dump(
    train_config,
    sort_keys=False,
    default_flow_style=False
)

display(Markdown(f"```yaml\n{yaml_text}\n```"))

您可以直接在 YAML 文件中修改参数或使用 CLI 参数覆盖它们。

#### 3.启动培训

要在 Qwen2.5-7B 的单个节点上运行训练，请使用以下命令：

In [ ]:
import subprocess

# Choose training mode: "bf16" or "fp8"
mode = "bf16"  # change to "fp8" if you want FP8 training

if mode == "bf16":
    cmd = """
    EXP=examples/megatron/configs/qwen2.5_7B-pretrain.yaml \
    bash examples/run_pretrain.sh \
        --train_iters 50
    """
elif mode == "fp8":
    cmd = """
    EXP=examples/megatron/configs/qwen2.5_7B-pretrain.yaml \
    bash examples/run_pretrain.sh \
        --train_iters 50 \
        --fp8 hybrid
    """
else:
    raise ValueError("Unsupported mode. Use 'bf16' or 'fp8'.")

print("Running command:\n", cmd)
subprocess.run(cmd, shell=True, check=True)

### 关键培训选项

以下关键选项适用于上面显示的训练命令：

**fp8**

* `hybrid` 启用 `FP8` GEMM。

**use_torch_fsdp2**

* `use_torch_fsdp2: 1` 启用torch FSDP-v2。
* 如果启用 FSDP，请将 `use_distributed_optimizer` 和 `overlap_param_gather` 设置为 `false`。

**个人资料**

* 要启用 PyTorch 分析，请设置以下参数：

  ``` yaml
  profile: true
  use_pytorch_profiler: true
  profile_step_end: 7
  profile_step_start: 6
  ```

**train_iters**

* 迭代总数（默认为`50`）。

**模拟数据**

* 默认为 `True`。

**微批量大小**

* 微批大小。

**全局批量大小**

* 全球 批大小。

**重新计算粒度**

* 用于激活检查点。

**层数**

* 用于减少层数，例如使用代理模型。


## 进一步阅读
- 有关 Primus 的介绍，请参阅 [Primus：AMD GPU 上大型模型的轻量级统一训练框架](https://rocm.blogs.amd.com/software-tools-optimization/primus/README.html)。
- 要了解如何结合使用 Primus 和 PyTorch 设置和运行训练，请参阅[使用 Primus 和 Pytorch TorchTitan 训练模型](https://rocm.docs.amd.com/projects/primus/en/latest/02-user-guide/torchtitan-training.html)。
- 要了解有关为 AMD Instinct MI300X 系列 GPU 配置系统的系统设置和管理实践的更多信息，请参阅[AMD Instinct MI300X 系统优化](https://instinct.docs.amd.com/projects/amdgpu-docs/en/latest/system-optimization/mi300x.html)。
- 有关使用 ROCm 进行 AI 的其他现成 Docker 映像的列表，请参阅 [AMD Infinity Hub](https://www.amd.com/en/developer/resources/infinity-hub.html#f-amd_hub_category=AI%20%26%20ML%20Models)。